Dataset usado:
 Insect Village Synthetic Dataset

In [1]:
import torch
import torchvision

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [6]:
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

PATH = 'C:/Users/aaran/Downloads/Insect Village Synthetic Dataset/Insect Classes/Insect Classes/'

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),(0.5, 0.5, 0.5))
])

full_ds = datasets.ImageFolder(PATH, transform=transform)

print(f"Clases detectadas: {full_ds.classes}")
print(f"Mapeo de etiquetas: {full_ds.class_to_idx}")

total_len = len(full_ds)
train_size = int(0.70 * total_len)
val_size = int(0.15 * total_len)
test_size = total_len - train_size - val_size

train_ds, val_ds, test_ds = random_split(full_ds, [train_size, val_size, test_size],
generator=torch.Generator().manual_seed(42))

train_dataloader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_dataloader = DataLoader(val_ds, batch_size=32, shuffle=False)
test_dataloader = DataLoader(test_ds, batch_size=32, shuffle=False)


Clases detectadas: ['Bees', 'Beetles', 'Butterfly', 'Cicada', 'Dragonfly', 'Grasshopper', 'Moth', 'Scorpion', 'Snail', 'Spider']
Mapeo de etiquetas: {'Bees': 0, 'Beetles': 1, 'Butterfly': 2, 'Cicada': 3, 'Dragonfly': 4, 'Grasshopper': 5, 'Moth': 6, 'Scorpion': 7, 'Snail': 8, 'Spider': 9}


En esta celda simplemente cargue el dataset con ImageFolder, cada clase esta en una subcarpeta asi que se carga por carpeta cada clase.
Luego preparo el dataloader para entrenar mediante batches.

In [25]:
from torch import nn

class CNN(nn.Module):
    """Some Information about CNN"""
    def __init__(self, n_canales=3, n_salidas=10):
        super(CNN, self).__init__()
        self.conv = torch.nn.Sequential(
            torch.nn.Conv2d(n_canales, 256, 3, 2, 2),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(4, 2)
        )
        self.fc = torch.nn.Linear(256 * 31 * 31, n_salidas)

    def forward(self, x):
        x = self.conv(x)
        x = torch.flatten(x, start_dim=1)
        x = self.fc(x)
        return x

Esta es mi red neuronal que voy a usar como se puede ver las capas principales son la convolucionar con la que ingreso los 3 canales rgb de cada imagen y tambien como cada imagen esta reescalada a 128 px para que el entrenamiento sea mas eficiente.
Luego reLu para introducir no linealidad
Luego la capa de pooling para reconstruir y obtener mas capas 
Finalmente la capa lineal que obtiene los 256 nuevos pixeles y obtiene cada una de las nuevas imagenes luego de que pasaran por todas las capas anteriores.
Aun no entiendo bien la covnersion de 256 * 31 * 31 pero tiene que ver con las transformaciones que se hicieron en las capas anteriores y tambien el n_salidas que seran las clases en este caso usaremos 10 por que tengo 10 clases.

In [26]:
model = CNN()
model

CNN(
  (conv): Sequential(
    (0): Conv2d(3, 256, kernel_size=(3, 3), stride=(2, 2), padding=(2, 2))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=4, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc): Linear(in_features=246016, out_features=10, bias=True)
)

In [ ]:
from tqdm import tqdm 
import numpy as np

def fit(model, epochs=50):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = torch.nn.CrossEntropyLoss()
    for epoch in range(1, epochs+1):
        model.train()
        train_loss, train_acc = [], []
        bar = tqdm(train_dataloader)
        for batch in bar:
            X, y = batch
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            y_hat = model(X)
            loss = criterion(y_hat, y)
            loss.backward()
            optimizer.step()
            train_loss.append(loss.item())
            acc = (y == torch.argmax(y_hat, axis=1)).sum().item() / len(y)
            train_acc.append(acc)
            bar.set_description(f"loss {np.mean(train_loss):.5f} acc {np.mean(train_acc):.5f}")
        bar = tqdm(test_dataloader)
        val_loss, val_acc = [], []
        model.eval()
        with torch.no_grad():
            for batch in bar:
                X, y = batch
                X, y = X.to(device), y.to(device)
                y_hat = model(X)
                loss = criterion(y_hat, y)
                val_loss.append(loss.item())
                acc = (y == torch.argmax(y_hat, axis=1)).sum().item() / len(y)
                val_acc.append(acc)
                bar.set_description(f"val_loss {np.mean(val_loss):.5f} val_acc {np.mean(val_acc):.5f}")
        print(f"Epoch {epoch}/{epochs} loss {np.mean(train_loss):.5f} val_loss {np.mean(val_loss):.5f} acc {np.mean(train_acc):.5f} val_acc {np.mean(val_acc):.5f}")

El ciclo de entrenamiento usa back propagation y criterio de optimizacion es adam y la funcion de perdida es CrossEntropyLoss que se usa para clasificacion mutlivariable que nos da una variable porcentual que hace que nos de una probabilidad de que pertenece a alguna clase.

In [55]:
model = CNN()
fit(model)

  0%|          | 0/219 [00:00<?, ?it/s]

val_loss 2.71538 val_acc 0.16841: 100%|██████████| 47/47 [00:03<00:00, 13.89it/s]


Epoch 1/20 loss 8.53063 val_loss 2.71538 acc 0.13875 val_acc 0.16841


val_loss 2.44309 val_acc 0.22074: 100%|██████████| 47/47 [00:02<00:00, 16.21it/s]


Epoch 2/20 loss 2.18177 val_loss 2.44309 acc 0.29775 val_acc 0.22074


...

Epoch 18/20 loss 0.18232 val_loss 4.82325 acc 0.96233 val_acc 0.21714


val_loss 5.04935 val_acc 0.22103: 100%|██████████| 47/47 [00:02<00:00, 16.23it/s]


Epoch 19/20 loss 0.15338 val_loss 5.04935 acc 0.97018 val_acc 0.22103


val_loss 5.24397 val_acc 0.22131: 100%|██████████| 47/47 [00:02<00:00, 16.15it/s]

Epoch 20/20 loss 0.12452 val_loss 5.24397 acc 0.97650 val_acc 0.22131


Luego de entrenar pruebo con imagenes nuevas y descargadas por mi que nada tuvieron que ver con el dataset

In [70]:
def probar(ruta):
    import torch
    from PIL import Image
    from torchvision import transforms

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Resize((128, 128)),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5,0.5, 0.5))
    ])

    imagen_pil = Image.open(ruta)
    imagen_tensor = transform(imagen_pil)

    input_batch = imagen_tensor.unsqueeze(0).to(device)


    model.eval()
    with torch.no_grad():
        outputs = model(input_batch)

        probabilidades = torch.softmax(outputs, dim=1)

        prediccion_idx = torch.argmax(probabilidades, dim=1).item()
        confianza = probabilidades[0][prediccion_idx].item()

    clases = full_ds.classes
    print(f"Prediccion: {clases[prediccion_idx]} ({confianza * 100:.2f}% de confianza)")

In [78]:
probar("C:/Users/aaran/Pictures/Machine Learning test imgs/insects/snail/snail2.jpg")
print("Acerto")

probar("C:/Users/aaran/Pictures/Machine Learning test imgs/insects/snail/snail1.jpg")
print("Se equivoco")

probar("C:/Users/aaran/Pictures/Machine Learning test imgs/insects/spider/spider1.jpg")
print("Acerto")

probar("C:/Users/aaran/Pictures/Machine Learning test imgs/insects/spider/spider3.jpg")
print("Se equivoco")

probar("C:/Users/aaran/Pictures/Machine Learning test imgs/insects/grasshoper/grasshoper1.jpg")
print("Se equivoco")

probar("C:/Users/aaran/Pictures/Machine Learning test imgs/insects/Butterfly/butterfly1.jpg")
print("Acerto")

probar("C:/Users/aaran/Pictures/Machine Learning test imgs/insects/Dragonfly/dragonfly3.jpg")
print("Acerto")

Prediccion: Snail (97.53% de confianza)
Acerto
Prediccion: Spider (66.97% de confianza)
Se equivoco
Prediccion: Spider (43.82% de confianza)
Acerto
Prediccion: Cicada (91.90% de confianza)
Se equivoco
Prediccion: Snail (74.89% de confianza)
Se equivoco
Prediccion: Butterfly (98.82% de confianza)
Acerto
Prediccion: Dragonfly (100.00% de confianza)
Acerto
